In [1]:
!pip install -Uqq fastbook

In [2]:
import fastbook
fastbook.setup_book()
from fastbook import *

In [3]:
from fastai.collab import *
from fastai.tabular.all import *

In [5]:
path = untar_data(URLs.ML_100k)
list(path.ls())

[Path('/root/.fastai/data/ml-100k/u.data'),
 Path('/root/.fastai/data/ml-100k/mku.sh'),
 Path('/root/.fastai/data/ml-100k/u.genre'),
 Path('/root/.fastai/data/ml-100k/allbut.pl'),
 Path('/root/.fastai/data/ml-100k/u5.base'),
 Path('/root/.fastai/data/ml-100k/ub.test'),
 Path('/root/.fastai/data/ml-100k/u1.base'),
 Path('/root/.fastai/data/ml-100k/u2.test'),
 Path('/root/.fastai/data/ml-100k/u.info'),
 Path('/root/.fastai/data/ml-100k/u3.base'),
 Path('/root/.fastai/data/ml-100k/u.item'),
 Path('/root/.fastai/data/ml-100k/ub.base'),
 Path('/root/.fastai/data/ml-100k/README'),
 Path('/root/.fastai/data/ml-100k/ua.test'),
 Path('/root/.fastai/data/ml-100k/u.user'),
 Path('/root/.fastai/data/ml-100k/u3.test'),
 Path('/root/.fastai/data/ml-100k/u4.base'),
 Path('/root/.fastai/data/ml-100k/u5.test'),
 Path('/root/.fastai/data/ml-100k/ua.base'),
 Path('/root/.fastai/data/ml-100k/u2.base'),
 Path('/root/.fastai/data/ml-100k/u4.test'),
 Path('/root/.fastai/data/ml-100k/u1.test'),
 Path('/root/.

In [6]:
with open(path/"README", "r") as t:
    print(t.read())

SUMMARY & USAGE LICENSE

MovieLens data sets were collected by the GroupLens Research Project
at the University of Minnesota.
 
This data set consists of:
	* 100,000 ratings (1-5) from 943 users on 1682 movies. 
	* Each user has rated at least 20 movies. 
        * Simple demographic info for the users (age, gender, occupation, zip)

The data was collected through the MovieLens web site
(movielens.umn.edu) during the seven-month period from September 19th, 
1997 through April 22nd, 1998. This data has been cleaned up - users
who had less than 20 ratings or did not have complete demographic
information were removed from this data set. Detailed descriptions of
the data file can be found at the end of this file.

Neither the University of Minnesota nor any of the researchers
involved can guarantee the correctness of the data, its suitability
for any particular purpose, or the validity of results based on the
use of the data set.  The data set may be used for any research
purposes under th

In [8]:
ratings = pd.read_csv(path/"u.data", delimiter="\t", names=["user", "movie", "rating", "timestamp"])
ratings.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [15]:
movies = pd.read_csv(path/"u.item", delimiter="|", usecols=(0,1), names=["movie", "title"], encoding="latin1")
movies.head()

,movie,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [16]:
ratings = ratings.merge(movies)
ratings.head()

,user,movie,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,63,242,3,875747190,Kolya (1996)
2,226,242,5,883888671,Kolya (1996)
3,154,242,3,879138235,Kolya (1996)
4,306,242,5,876503793,Kolya (1996)


In [17]:
dls = CollabDataLoaders.from_df(ratings, item_name='title', bs=64)
dls.show_batch()

,user,title,rating
0,542,My Left Foot (1989),4
1,422,Event Horizon (1997),3
2,311,"African Queen, The (1951)",4
3,595,Face/Off (1997),4
4,617,Evil Dead II (1987),1
5,158,Jurassic Park (1993),5
6,836,Chasing Amy (1997),3
7,474,Emma (1996),3
8,466,Jackie Chan's First Strike (1996),3
9,554,Scream (1996),3


In [20]:
embs = get_emb_sz(dls)
embs

[(944, 74), (1665, 102)]

In [24]:
class CollabNN(Module):
    def __init__(self, user_sz, item_sz, y_range=(0, 5.5), n_act=100):
        self.user_factors = Embedding(*user_sz)
        self.item_factors = Embedding(*item_sz)
        self.layers = nn.Sequential(
            nn.Linear((user_sz[1] + item_sz[1]), n_act),
            nn.ReLU(),
            nn.Linear(n_act, 1)
        )
        self.y_range = y_range
        
    def forward(self, x):
        users = self.user_factors(x[:,0])
        items = self.item_factors(x[:,1])
        fs = torch.cat((users, items), dim=1)
        x = self.layers(fs)
        return sigmoid_range(x, *self.y_range)

In [26]:
model = CollabNN(*embs)
model

CollabNN(
  (user_factors): Embedding(944, 74)
  (item_factors): Embedding(1665, 102)
  (layers): Sequential(
    (0): Linear(in_features=176, out_features=100, bias=True)
    (1): ReLU()
    (2): Linear(in_features=100, out_features=1, bias=True)
  )
)

In [28]:
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3, wd=0.01)

epoch,train_loss,valid_loss,time
0,0.814850,0.888835,00:09
1,0.830918,0.871842,00:09
2,0.788531,0.866448,00:09
3,0.731804,0.857170,00:09
4,0.689824,0.868318,00:09
